# 📅 Sample Data Daily Refresh (Scheduled Task)

## Purpose
Ensures business date columns in `miqsadata` always reflect data relative to today,
anchored to the fixed `BASE_DATE` of `2026-04-30`.

## Idempotency Strategy
**Dates in the Lakehouse are always expressed as:** `original_value + (today - BASE_DATE)`

On each run, the notebook:
1. Checks if a successful refresh has already run **today** — if yes, it **exits cleanly**.
2. Computes today's delta = `today - BASE_DATE`
3. Computes yesterday's delta = `yesterday - BASE_DATE`
4. Applies an **incremental shift** of exactly **1 day** (today_delta - yesterday_delta)
   so dates advance by 1 day each calendar day — never cumulatively over-shifted.

## Safety Controls
- ✅ Idempotent — safe to re-run multiple times on the same day
- ✅ NULL dates always preserved
- ✅ Execution log tracks every run
- ✅ Dry-run mode available
- ✅ Pre/post validation on each column

## 🔄 Recommended Setup
Schedule this notebook as a daily pipeline job (e.g., 2:00 AM UTC).

---

## 🔧 PARAMETERS — Modify before first run

In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

BASE_DATE = "2026-04-30"   # Fixed anchor date — never change after initial run
DRY_RUN   = False          # Set True to preview without writing
FORCE_RUN = False          # Set True to bypass the idempotency check (use with care)
CATALOG   = "miqsadata"   # Fabric Lakehouse catalog name
LOG_TABLE = "default.date_refresh_execution_log"

print(f"▶️  BASE_DATE  : {BASE_DATE}")
print(f"▶️  DRY_RUN    : {DRY_RUN}")
print(f"▶️  FORCE_RUN  : {FORCE_RUN}")
print(f"▶️  CATALOG    : {CATALOG}")
print(f"▶️  LOG_TABLE  : {LOG_TABLE}")

## 📋 Configuration — Schemas, Tables, and Columns to Update

In [ ]:
# ============================================================
# INCLUDE_CONFIG
# Schema → Table → [columns to UPDATE]
# Only DATE-typed columns approved for shifting are listed here.
# ============================================================

INCLUDE_CONFIG = {
    "sales": {
        "Order": ["OrderDate"],
    },
    "finance": {
        "invoice": ["InvoiceDate", "DueDate"],
        "payment": ["PaymentDate"],
    },
    "inventory": {
        "InventoryTransactions": ["TransactionDate", "CreatedDate"],
        "Inventory": ["LastUpdated"],
        "PurchaseOrders": ["OrderDate", "ExpectedDeliveryDate", "ActualDeliveryDate", "CreatedDate"],
        "PurchaseOrderItems": ["ExpectedDate", "ReceivedDate", "CreatedDate"],
        "DemandForecast": ["ForecastDate", "CreatedDate"],
    },
    "supplychain": {
        "SupplyChainEvents": ["StartDate", "EndDate", "CreatedDate"],
        "SupplyChainEventImpacts": ["CreatedDate"],
    },
    "product": {
        "Product": ["SellStartDate", "SellEndDate"],
    },
}

# ============================================================
# EXCLUDE_CONFIG
# Columns that must NEVER be updated (safety guard)
# ============================================================

EXCLUDE_CONFIG = {
    "customer": {"Customer": ["DateOfBirth", "CustomerEstablishedDate"]},
    "finance": {"account": ["CreatedDate", "ClosedDate"]},
    "product": {"Product": ["CreatedDate", "UpdatedDate"]},
    "inventory": {"Inventory": ["CreatedDate"]},
    "supplychain": {"Suppliers": ["CreatedDate"]},
    "shared": {"DimDate": ["DateKey"]},
}

# ============================================================
# SKIP_SCHEMAS
# Schemas that are entirely skipped
# ============================================================

SKIP_SCHEMAS = ["customer", "shared"]

print("✅ Configuration loaded")
print(f"   Included: {sum(len(tables) for tables in INCLUDE_CONFIG.values())} tables")
print(f"   Excluded: {sum(len(cols) for schema in EXCLUDE_CONFIG.values() for cols in schema.values())} columns")

## 🛠️ Helper Functions

In [ ]:
# ============================================================
# HELPER FUNCTIONS (same as Notebook 01)
# ============================================================

from datetime import date, datetime, timedelta
import time
import uuid
from pyspark.sql import functions as F

def calculate_delta(base_date_str: str) -> int:
    """Calculate days between today and base date."""
    base = date.fromisoformat(base_date_str)
    today = date.today()
    return (today - base).days

def get_run_id() -> str:
    """Generate unique run ID."""
    ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    uid = str(uuid.uuid4())[:8]
    return f"RUN_{ts}_{uid}"

def table_exists(schema: str, table: str) -> bool:
    """Check if table exists."""
    try:
        spark.sql(f"DESCRIBE TABLE {schema}.{table}")
        return True
    except:
        return False

def column_exists(schema: str, table: str, column: str) -> bool:
    """Check if column exists."""
    try:
        cols = [c.name.lower() for c in spark.table(f"{schema}.{table}").schema]
        return column.lower() in cols
    except:
        return False

def is_column_excluded(schema: str, table: str, column: str) -> bool:
    """Check if column is in exclude list."""
    if schema in EXCLUDE_CONFIG:
        if table in EXCLUDE_CONFIG[schema]:
            return column in EXCLUDE_CONFIG[schema][table]
    return False

def get_non_null_count(schema: str, table: str, column: str) -> int:
    """Get count of non-NULL values in a column."""
    try:
        return spark.sql(f"SELECT COUNT({column}) AS cnt FROM {schema}.{table}").collect()[0]["cnt"]
    except:
        return -1

def capture_date_column_stats(schema: str, table: str, column: str) -> dict:
    """Capture min, max, null stats for a date column."""
    try:
        row = spark.sql(f"""
            SELECT
                MIN({column})                          AS min_date,
                MAX({column})                          AS max_date,
                SUM(CASE WHEN {column} IS NULL THEN 1 ELSE 0 END) AS null_count,
                COUNT({column})                        AS non_null_count,
                COUNT(*)                               AS total_rows
            FROM {schema}.{table}
        """).collect()[0]
        return {
            "schema": schema,
            "table": table,
            "column": column,
            "min_date": str(row["min_date"]),
            "max_date": str(row["max_date"]),
            "null_count": int(row["null_count"]),
            "non_null_count": int(row["non_null_count"]),
            "total_rows": int(row["total_rows"]),
        }
    except Exception as e:
        return {"schema": schema, "table": table, "column": column, "error": str(e)}

def validate_shift(before: dict, after: dict, delta_days: int) -> tuple:
    """Validate that date shift was applied correctly."""
    if "error" in before or "error" in after:
        return False, "Stats capture failed"
    
    errors = []
    if before["null_count"] != after["null_count"]:
        errors.append(f"NULL count changed: {before['null_count']} → {after['null_count']}")
    if before["non_null_count"] != after["non_null_count"]:
        errors.append(f"Row count changed: {before['non_null_count']} → {after['non_null_count']}")
    
    return len(errors) == 0, " | ".join(errors) if errors else "OK"

def ensure_log_table():
    """Create execution log table if it doesn't exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
            run_id            STRING    NOT NULL,
            run_date          TIMESTAMP NOT NULL,
            notebook          STRING,
            base_date         STRING,
            delta_days        INT,
            schema_name       STRING,
            table_name        STRING,
            column_name       STRING,
            rows_affected     LONG,
            status            STRING,
            error_message     STRING,
            duration_seconds  DOUBLE,
            validation_passed STRING
        )
        USING DELTA
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """)

def log_column_update(run_id: str, schema: str, table: str, column: str, rows: int, delta: int, duration: float, status: str, error: str = None):
    """Write log record for column update."""
    spark.sql(f"""
        INSERT INTO {LOG_TABLE}
        VALUES ('{run_id}', current_timestamp(), 'notebook_02_daily', '{BASE_DATE}', {delta}, 
                '{schema}', '{table}', '{column}', {rows}, '{status}', '{error or ""}', {duration}, 'N/A')
    """)

def get_all_targets() -> list:
    """Get list of (schema, table, column) tuples to process."""
    targets = []
    for schema, tables in INCLUDE_CONFIG.items():
        if schema in SKIP_SCHEMAS:
            continue
        for table, columns in tables.items():
            for column in columns:
                targets.append((schema, table, column))
    return targets

print("✅ Helper functions loaded")

## 📐 STEP 1 — Calculate Today's and Yesterday's Delta

In [ ]:
today = date.today()
yesterday = today - timedelta(days=1)
base = date.fromisoformat(BASE_DATE)

today_delta = (today - base).days
yesterday_delta = (yesterday - base).days
incremental_shift = today_delta - yesterday_delta   # Always 1

print(f"  📅 BASE_DATE         : {base}")
print(f"  📅 Today             : {today}")
print(f"  📅 Yesterday         : {yesterday}")
print(f"  📐 Today delta       : {today_delta} days since base")
print(f"  📐 Yesterday delta   : {yesterday_delta} days since base")
print(f"  📐 Incremental shift : {incremental_shift} day(s) to apply today")

## 🛡️ STEP 2 — Guard: Validate Delta

In [ ]:
if today_delta <= 0:
    raise ValueError(
        f"❌ Today ({today}) is not after BASE_DATE ({BASE_DATE}). "
        f"Delta = {today_delta}. Aborting."
    )

print(f"✅ Delta check passed: {today_delta} total days since base")

## 📋 STEP 3 — Initialize Logging

In [ ]:
ensure_log_table()
run_id = get_run_id()
print(f"  🆔 Run ID : {run_id}")
print(f"  📋 Log Table: {LOG_TABLE}")

## 🔄 STEP 4 — Idempotency Check

In [ ]:
if FORCE_RUN:
    print("⚡ FORCE_RUN = True — bypassing idempotency check")
else:
    try:
        already_ran_today = spark.sql(f"""
            SELECT COUNT(*) AS cnt
            FROM   {LOG_TABLE}
            WHERE  notebook  = 'notebook_02_daily'
              AND  status    = 'Success'
              AND  DATE(run_date) = CURRENT_DATE()
        """).collect()[0]["cnt"]

        if already_ran_today > 0:
            print(f"\n✅ Already refreshed today ({today}). Nothing to do.")
            print("   To force a re-run, set FORCE_RUN = True")
            dbutils.notebook.exit("Already refreshed today — no-op")
        else:
            print(f"✅ No successful run found for today ({today}) — proceeding with refresh")

    except Exception as e:
        # If log table doesn't exist yet this is the first run — safe to proceed
        print(f"  ℹ️  Could not query log table (likely first run): {e}")
        print("  Proceeding with refresh...")

## 📸 STEP 5 — Pre-Update Validation

In [ ]:
print("\n📸 Capturing pre-update statistics...\n")

pre_stats = {}
targets = get_all_targets()

for schema, table, column in targets:
    if not table_exists(schema, table):
        continue
    if not column_exists(schema, table, column):
        continue
    stats = capture_date_column_stats(schema, table, column)
    pre_stats[(schema, table, column)] = stats
    print(f"  ✅ {schema}.{table}.{column:30} | Min: {stats['min_date'][:10]} | Max: {stats['max_date'][:10]} | Non-NULL: {stats['non_null_count']:,}")

print(f"\n  ✅ Pre-stats captured for {len(pre_stats)} columns")

## 🔄 STEP 6 — Main Update Loop (Incremental Shift)

In [ ]:
print(f"\n🔄 Starting daily refresh — shifting by {incremental_shift} day(s)\n")

results = []

for schema, table, column in targets:

    key = (schema, table, column)
    label = f"{schema}.{table}.{column}"
    t_start = time.time()

    try:
        if not table_exists(schema, table):
            print(f"  ⏭️  SKIP  {label} — table not found")
            log_column_update(run_id, schema, table, column, 0, incremental_shift, time.time() - t_start, "Skipped", "Table not found")
            results.append({"label": label, "status": "Skipped", "rows": 0})
            continue

        if not column_exists(schema, table, column):
            print(f"  ⏭️  SKIP  {label} — column not found")
            log_column_update(run_id, schema, table, column, 0, incremental_shift, time.time() - t_start, "Skipped", "Column not found")
            results.append({"label": label, "status": "Skipped", "rows": 0})
            continue

        if is_column_excluded(schema, table, column):
            print(f"  🛑 BLOCKED {label} — force-excluded")
            log_column_update(run_id, schema, table, column, 0, incremental_shift, time.time() - t_start, "Skipped", "Force-excluded")
            results.append({"label": label, "status": "Blocked", "rows": 0})
            continue

        non_null_rows = get_non_null_count(schema, table, column)

        if DRY_RUN:
            duration = time.time() - t_start
            print(f"  🔵 DRYRUN {label:<55} → would shift {non_null_rows:>8,} rows by {incremental_shift}d")
            log_column_update(run_id, schema, table, column, non_null_rows, incremental_shift, duration, "DryRun")
            results.append({"label": label, "status": "DryRun", "rows": non_null_rows})
            continue

        # LIVE UPDATE: efficient Delta SQL UPDATE (incremental 1-day shift)
        spark.sql(f"""
            UPDATE {schema}.{table}
            SET    {column} = DATE_ADD({column}, {incremental_shift})
            WHERE  {column} IS NOT NULL
        """)

        duration = time.time() - t_start
        print(f"  ✅ OK     {label:<55} → shifted {non_null_rows:>8,} rows in {duration:.1f}s")

        log_column_update(run_id, schema, table, column, non_null_rows, incremental_shift, duration, "Success")
        results.append({"label": label, "status": "Success", "rows": non_null_rows})

    except Exception as e:
        duration = time.time() - t_start
        err_msg = str(e)[:200]
        print(f"  ❌ FAIL  {label} — {err_msg}")
        log_column_update(run_id, schema, table, column, 0, incremental_shift, duration, "Failed", err_msg)
        results.append({"label": label, "status": "Failed", "rows": 0})

print(f"\n✅ Update loop complete — {len(results)} columns processed")

## 🔍 STEP 7 — Post-Update Validation

In [ ]:
if DRY_RUN:
    print("ℹ️  DRY RUN — skipping post-validation (no changes were written)")
else:
    print("\n🔍 Running post-update validation...\n")

    validation_failures = []

    for schema, table, column in targets:
        key = (schema, table, column)
        if key not in pre_stats:
            continue

        post_stat = capture_date_column_stats(schema, table, column)
        passed, message = validate_shift(pre_stats[key], post_stat, incremental_shift)

        status_icon = "✅" if passed else "❌"
        print(f"  {status_icon} {schema}.{table}.{column} — {message}")

        if not passed:
            validation_failures.append(f"{schema}.{table}.{column}: {message}")

    if validation_failures:
        print(f"\n⚠️  {len(validation_failures)} validation failure(s)")
        for f in validation_failures:
            print(f"     • {f}")
    else:
        print(f"\n✅ All validations passed!")

## 📊 STEP 8 — Execution Summary

In [ ]:
success_count = sum(1 for r in results if r["status"] == "Success")
failed_count = sum(1 for r in results if r["status"] == "Failed")
skipped_count = sum(1 for r in results if r["status"] in ("Skipped", "Blocked"))
dryrun_count = sum(1 for r in results if r["status"] == "DryRun")
total_rows = sum(r["rows"] for r in results)

print("\n" + "=" * 70)
print("  📊 EXECUTION SUMMARY — Notebook 02 Daily Refresh")
print("=" * 70)
print(f"  Run ID           : {run_id}")
print(f"  Base Date        : {BASE_DATE}")
print(f"  Today            : {today}")
print(f"  Incremental Shift: {incremental_shift} day(s)")
print(f"  Dry Run          : {DRY_RUN}")
print(f"  ✅ Successful     : {success_count}")
print(f"  ❌ Failed         : {failed_count}")
print(f"  ⏭️  Skipped        : {skipped_count}")
print(f"  🔵 Dry Run        : {dryrun_count}")
print(f"  📦 Total Rows     : {total_rows:,}")
print("=" * 70)
print(f"\n✅ Execution Log: {LOG_TABLE}")
print(f"   Query: SELECT * FROM {LOG_TABLE} WHERE run_id = '{run_id}'")

---
## ✅ Notebook 02 Complete

Daily refresh has been applied. All approved business date columns have been shifted by 1 day.

| Detail | Value |
|--------|-------|
| Execution Log | `default.date_refresh_execution_log` |
| For initial setup | Use `01_sample_data_initial_refresh` |
| Safe to re-run? | ✅ Yes (idempotency check prevents duplicate shifts) |

> **🔄 Scheduled:** Run this notebook daily (recommended: 2:00 AM UTC) as a Fabric pipeline job.